# Chronos-2 ReNile-IOT Evaluation

This notebook evaluates Chronos-2 using only ReNile-IOT data.

- Fetch data from `2026-05-01 00:00` onward.
- Use `2026-05-01 00:00` through `2026-05-14 23:00` as the 336-hour context.
- Forecast `2026-05-15 00:00` through `2026-05-21 23:00` with a 168-hour horizon.
- Interpolation is used only to prepare the model context.
- Metrics are computed only against raw ReNile-IOT actual observations, never interpolated actuals.

## Imports

In [ ]:
from __future__ import annotations

from getpass import getpass
import math
import os
from pathlib import Path
import sys
import time

import httpx
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.core.config import get_settings
from src.core.logging import configure_logging
from src.services.weather.providers.renile_iot import parse_renile_iot_payload

## Configuration

Set `RENILE_IOT_JWT` and `RENILE_IOT_DEVICE_ID` in your environment, or enter them when prompted.

In [ ]:
settings = get_settings()
configure_logging(settings.logging_level)

JWT = os.getenv("RENILE_IOT_JWT") or getpass("ReNile-IOT JWT: ")
DEVICE_ID = os.getenv("RENILE_IOT_DEVICE_ID") or input("ReNile-IOT device_id: " ).strip()

FETCH_START = pd.Timestamp("2026-05-01 00:00")
CONTEXT_START = pd.Timestamp("2026-05-01 00:00")
CONTEXT_END = pd.Timestamp("2026-05-14 23:00")
FORECAST_START = pd.Timestamp("2026-05-15 00:00")
FORECAST_END = pd.Timestamp("2026-05-21 23:00")
CONTEXT_HOURS = 336
PREDICTION_LENGTH = 168

MODEL_ID = settings.chronos_model_id
DEVICE_MAP = settings.chronos_device_map if torch.cuda.is_available() else "cpu"

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE_MAP}")
print(f"Context: {CONTEXT_START} -> {CONTEXT_END} ({CONTEXT_HOURS} hours)")
print(f"Forecast: {FORECAST_START} -> {FORECAST_END} ({PREDICTION_LENGTH} hours)")

## Fetch Raw ReNile-IOT Data

In [ ]:
def fetch_renile_iot_payload() -> dict:
    params = {
        "data_type": settings.renile_iot_data_type,
        "start_time": FETCH_START.strftime("%Y-%m-%d %H:%M"),
        "device_id": DEVICE_ID,
    }
    headers = {
        "Authorization": f"JWT {JWT}",
        "Accept": "application/json",
    }
    with httpx.Client(timeout=settings.renile_iot_timeout_seconds) as client:
        response = client.get(settings.renile_iot_url, params=params, headers=headers)
        response.raise_for_status()
    return response.json()


payload = fetch_renile_iot_payload()
raw_df = parse_renile_iot_payload(payload)
raw_df = raw_df[(raw_df["timestamp"] >= CONTEXT_START) & (raw_df["timestamp"] <= FORECAST_END)].copy()
raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)

print(f"Raw rows: {len(raw_df)}")
print(f"Sensors: {raw_df.columns.drop('timestamp').tolist()}")
print(f"Range: {raw_df['timestamp'].min()} -> {raw_df['timestamp'].max()}")
display(raw_df.head())
display(raw_df.tail())

## Build Model Context

The model context must be regular hourly data. Missing context values are interpolated here only. The forecast-period actuals remain raw and are not interpolated.

In [ ]:
context_raw_df = raw_df[(raw_df["timestamp"] >= CONTEXT_START) & (raw_df["timestamp"] <= CONTEXT_END)].copy()
actual_raw_df = raw_df[(raw_df["timestamp"] >= FORECAST_START) & (raw_df["timestamp"] <= FORECAST_END)].copy()

targets = [column for column in raw_df.columns if column != "timestamp"]
targets = [
    target
    for target in targets
    if context_raw_df[target].notna().any() and actual_raw_df[target].notna().any()
]
if not targets:
    raise ValueError("No sensors have both context data and raw actual forecast-period data.")

context_hourly_df = (
    context_raw_df[["timestamp", *targets]]
    .drop_duplicates(subset=["timestamp"], keep="last")
    .sort_values("timestamp")
    .set_index("timestamp")
    .asfreq("h")
)
context_hourly_df = context_hourly_df.interpolate(method="time").ffill().bfill()
context_hourly_df = context_hourly_df.loc[CONTEXT_START:CONTEXT_END].reset_index()

if len(context_hourly_df) != CONTEXT_HOURS:
    raise ValueError(f"Expected {CONTEXT_HOURS} context hours, got {len(context_hourly_df)}.")
if context_hourly_df[targets].isna().any().any():
    raise ValueError("Context still contains missing values after interpolation.")

chronos_context = pd.DataFrame({"item_id": "weather_series", "timestamp": context_hourly_df["timestamp"]})
for target in targets:
    chronos_context[target] = context_hourly_df[target].astype(float)

print(f"Targets used: {targets}")
print(f"Context rows: {len(chronos_context)}")
print(f"Raw actual rows before melt: {len(actual_raw_df)}")
display(chronos_context.head())
display(chronos_context.tail())

## Forecast

In [ ]:
load_start = time.perf_counter()
pipeline = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEVICE_MAP)
load_seconds = time.perf_counter() - load_start

forecast_start_time = time.perf_counter()
predictions_df = pipeline.predict_df(
    chronos_context,
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.1, 0.5, 0.9],
    timestamp_column="timestamp",
    target=targets,
)
inference_seconds = time.perf_counter() - forecast_start_time

predictions_df = predictions_df.rename(
    columns={"predictions": "prediction", "0.1": "q10", "0.5": "q50", "0.9": "q90", 0.1: "q10", 0.5: "q50", 0.9: "q90"}
)
predictions_df["timestamp"] = pd.to_datetime(predictions_df["timestamp"])
if "target_name" not in predictions_df.columns:
    if len(targets) != 1:
        raise ValueError("Chronos output is missing target_name for multi-target predictions.")
    predictions_df["target_name"] = targets[0]
predictions_df = predictions_df.sort_values(["target_name", "timestamp"]).reset_index(drop=True)

print(f"Model load seconds: {load_seconds:.2f}")
print(f"Inference seconds: {inference_seconds:.2f}")
print(f"Prediction rows: {len(predictions_df)}")
display(predictions_df.head())
display(predictions_df.tail())

## Compare With Raw ReNile-IOT Actuals

This section drops missing raw actuals and merges predictions only with timestamps that were actually returned by ReNile-IOT. No interpolated actual values are used.

In [ ]:
actual_long = actual_raw_df[["timestamp", *targets]].melt(
    id_vars=["timestamp"],
    var_name="target_name",
    value_name="actual",
).dropna(subset=["actual"])
actual_long["timestamp"] = pd.to_datetime(actual_long["timestamp"])
actual_long["actual"] = pd.to_numeric(actual_long["actual"], errors="coerce")
actual_long = actual_long.dropna(subset=["actual"])

comparison_df = actual_long.merge(
    predictions_df[["timestamp", "target_name", "prediction", "q10", "q50", "q90"]],
    on=["timestamp", "target_name"],
    how="inner",
)

if comparison_df.empty:
    raise ValueError("No raw ReNile-IOT actual timestamps matched Chronos prediction timestamps.")

expected_prediction_timestamps = pd.date_range(FORECAST_START, FORECAST_END, freq="h")
actual_matched_timestamps = comparison_df["timestamp"].nunique()
print(f"Raw actual values available for scoring: {len(actual_long)}")
print(f"Prediction/actual comparison rows: {len(comparison_df)}")
print(f"Matched forecast timestamps: {actual_matched_timestamps} / {len(expected_prediction_timestamps)}")
display(comparison_df.head())
display(comparison_df.tail())

## Metrics

For each target, `error = prediction - actual`.

- `mae`: mean absolute error.
- `mse`: mean squared error.
- `rmse`: square root of MSE.
- `bias`: mean error; positive means overprediction, negative means underprediction.
- `q10_q90_coverage_percent`: percent of raw actual values inside Chronos' 10%-90% interval.

In [ ]:
def safe_mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    mask = np.abs(actual) > 1e-8
    if not mask.any():
        return math.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


def r2_score(actual: np.ndarray, predicted: np.ndarray) -> float:
    ss_res = float(np.sum((actual - predicted) ** 2))
    ss_tot = float(np.sum((actual - actual.mean()) ** 2))
    if ss_tot == 0:
        return math.nan
    return 1 - (ss_res / ss_tot)


def is_direction_target(target_name: str) -> bool:
    return "direction" in target_name.lower()


def angular_error_degrees(predicted: np.ndarray, actual: np.ndarray) -> np.ndarray:
    return ((predicted - actual + 180) % 360) - 180


rows = []
for target_name, group in comparison_df.groupby("target_name", sort=True):
    actual = group["actual"].to_numpy(dtype=float)
    predicted = group["prediction"].to_numpy(dtype=float)

    if is_direction_target(target_name):
        errors = angular_error_degrees(predicted, actual)
        mape = math.nan
        r2 = math.nan
    else:
        errors = predicted - actual
        mape = safe_mape(actual, predicted)
        r2 = r2_score(actual, predicted)

    mse = float(np.mean(errors ** 2))
    coverage = ((group["actual"] >= group["q10"]) & (group["actual"] <= group["q90"])).mean() * 100
    rows.append(
        {
            "target_name": target_name,
            "raw_actual_points": int(len(group)),
            "mae": float(np.mean(np.abs(errors))),
            "mse": mse,
            "rmse": float(np.sqrt(mse)),
            "bias": float(errors.mean()),
            "mape_percent": mape,
            "r2": r2,
            "q10_q90_coverage_percent": float(coverage),
        }
    )

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

## Runtime Summary

In [ ]:
runtime_summary = {
    "model_id": MODEL_ID,
    "device_map": DEVICE_MAP,
    "context_hours": CONTEXT_HOURS,
    "prediction_length": PREDICTION_LENGTH,
    "targets_count": len(targets),
    "prediction_rows": len(predictions_df),
    "comparison_rows_raw_actual_only": len(comparison_df),
    "model_load_seconds": load_seconds,
    "inference_seconds": inference_seconds,
    "inference_ms_per_prediction_row": inference_seconds * 1000 / len(predictions_df),
    "inference_ms_per_target": inference_seconds * 1000 / len(targets),
}
display(pd.DataFrame([runtime_summary]).round(4))